# 🎯 Module 3 — Class 4 Homework: Feature Selection

**Topic:** Feature selection — mutual_info, correlation, RFE

## 📋 What you have to do

This notebook **already has working code**. Your job:

1. **Run every cell from top to bottom** — it should run with no errors.
2. **Add a comment on EVERY code cell** that explains what the cell does, in your own words.
   - Use `#` to write comments inside the code cell.
   - Write at least 1 short sentence per cell. Longer is better.
3. **Save your notebook** as `Module<X>_Class<Y>_<YourName>.ipynb` and submit.

**Example of a good commented cell:**

```python
# Count how many customers churned vs stayed
# value_counts() returns the number of rows for each unique value
df['Churn'].value_counts()
```

**Example of a BAD comment (do not do this):**

```python
# count
df['Churn'].value_counts()
```

---


In [ ]:
# === SETUP — run this first ===
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings; warnings.filterwarnings('ignore')

url = 'https://raw.githubusercontent.com/IBM/telco-customer-churn-on-icp4d/master/data/Telco-Customer-Churn.csv'
df = pd.read_csv(url)
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce').fillna(0)
print('Loaded:', df.shape)


### Cell 1 — prepare X (features) and y (target)

In [ ]:
df['Churn_bin'] = df['Churn'].map({'No': 0, 'Yes': 1})
num_cols = ['tenure', 'MonthlyCharges', 'TotalCharges', 'SeniorCitizen']
X = df[num_cols]
y = df['Churn_bin']
print('X shape:', X.shape, '| y shape:', y.shape)

### Cell 2 — correlation between each feature and target

In [ ]:
corr = X.copy()
corr['Churn'] = y
corr.corr()['Churn'].sort_values(ascending=False)

### Cell 3 — mutual information scores

In [ ]:
from sklearn.feature_selection import mutual_info_classif
mi = mutual_info_classif(X, y, random_state=42)
mi_scores = pd.Series(mi, index=X.columns).sort_values(ascending=False)
mi_scores.round(4)

### Cell 4 — bar chart of mutual information scores

In [ ]:
plt.figure(figsize=(7, 4))
mi_scores.plot(kind='barh', color='steelblue')
plt.title('Mutual Information with Churn')
plt.xlabel('MI score')
plt.tight_layout()
plt.show()

### Cell 5 — RFE: pick the top 2 features using LogisticRegression

In [ ]:
from sklearn.feature_selection import RFE
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
X_scaled = StandardScaler().fit_transform(X)
rfe = RFE(LogisticRegression(max_iter=1000), n_features_to_select=2)
rfe.fit(X_scaled, y)
result = pd.DataFrame({'feature': X.columns, 'kept': rfe.support_, 'rank': rfe.ranking_})
result.sort_values('rank')

### Cell 6 — train a model with ALL features vs top 2 features

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2, random_state=42)
m_all = LogisticRegression(max_iter=1000).fit(X_train, y_train)
print('All features accuracy:', round(accuracy_score(y_test, m_all.predict(X_test)), 4))
top_idx = np.where(rfe.support_)[0]
m_top = LogisticRegression(max_iter=1000).fit(X_train[:, top_idx], y_train)
print('Top 2 features accuracy:', round(accuracy_score(y_test, m_top.predict(X_test[:, top_idx])), 4))

---

## 📤 Submit

Comment every cell. Push to `module-3/class_4/submissions/<YourName>/`.